# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [2]:
# set up environment

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')


In [3]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [8]:
# Get gpt-4o-mini to answer, with streaming

client = OpenAI()
messages = [{"role":"user", "content":question}]

response = client.chat.completions.create(
    model = MODEL_GPT,
    messages = messages,
)

display(Markdown(response.choices[0].message.content))

The code you provided is using Python's `yield from` statement in conjunction with a set comprehension. Let's break it down to understand what it does.

### Breakdown of the Code

1. **Set Comprehension**:
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   - This part creates a set of authors from a list of `books`.
   - It iterates over each `book` in the `books` iterable (which should be an iterable of dictionaries representing individual book details).
   - For each `book`, it calls `book.get("author")` to retrieve the value associated with the key `"author"`.
   - The `if book.get("author")` condition filters out any `book` entries where the author is `None` or an empty string, thereby ensuring that only valid authors are included in the set.

2. **`yield from` Statement**:
   ```python
   yield from {...}
   ```
   - The `yield from` statement is used in Python generators. It allows you to yield all items from an iterable (in this case, the set of authors) one by one.
   - When this code is executed inside a generator function, it will yield each unique author found in the `books` collection.

### Overall Functionality

- The whole code snippet is designed to iterate through a collection of `books`, extract the unique authors from them, and yield each author one at a time, filtering out any entries that do not have a valid author.
- The result is that if this code is part of a generator function, when called, it will produce (yield) each unique author for further processing, and any duplicate author entries will be automatically handled by the set's properties.

### Example Usage
Here's an example implementation:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Example usage
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"},  # No author
]

for author in get_unique_authors(books):
    print(author)
```

### Output
```
Author A
Author B
```

In this example, `get_unique_authors` will yield each unique author, and duplicate authors will only appear once in the output. Authors with no name are filtered out.

In [12]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"

llama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") 
response = llama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = messages,
)

display(Markdown(response.choices[0].message.content))

This line of code is using a few features of Python generators to extract specific data from a list of dictionaries.

Here's a breakdown:

- `yield from`: This keyword allows you to delegate the creation of an iterator to another iterable (in this case, a generator expression).

- `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension that iterates over each item (`book`) in the `books` list and filters out any items that don't have an `"author"` key.

The `yield from` part tells Python to create a new iterator (an iterable) that will iterate over each of these filtered items. But here's where it gets interesting: for each item, it calls the `get("author")` method on the book dictionary and adds the result to the new iterable.

So overall, this line of code is creating an iterable sequence of just the authors from your list of books, but only if those books have an `"author"` key. 

Here's a more step-by-step breakdown:

1. Iterate over each item (`book`) in the `books` list.
2. For each book:
    - If it has an `"author"` key
        - Get its author name using `book.get("author")`
        - Add that to the new iterable

The end result is an iterator object (the generator) that yields these authors one by one.

Here's an example of what this might look like in practice:
```python
books = [
    {"title": "Title A", "author": "Author 1"},
    {"title": "Title B"}
]

# Generate a list of only the books with authors. This requires converting the yield to a list (because the original
# expression was a generator).
authors = [book["author"] for book in books if "author" in book]
print(authors) # Outputs: ['Author 1']

for author in authors:
    print(author)
```
This `yield from` trick allows us to elegantly extract this list of author names, all without having to explicitly loop over the original `books`, like so:

```python
# Doing it as above:
authors = []
for book in books:
    if "author" in book:
        authors.append(book["author"])
print(authors) # Outputs: ['Author 1']

for author in authors:
    print(author)
```